# QPSK-like 11x15 raw CC matrix generation from `generated_boi_p96_fft2_pt/`

Reads `generated_boi_p96_fft2_pt/{dataset}_batch_NN_boi_p96_fft2_full.pt` (produced by
`pre_BOI+FFT-Generate-2022-compact-vectors-final.ipynb`) and, for every QPSK-like
sample (`qpsk`/`16qam`/`64qam`/`256qam`) with `status=True`, builds a raw (no abs/warp/scale)
complex64 `[N,1,11,15]` cyclic-cumulant matrix from `x_processed` (the UNCLIPPED BOI output --
verified this session to give better downstream classification accuracy than the P96-clipped
`z`), using:

- **Rs**: `rs_estimate_prominence_corrected` (Method 2's blind circular-prominence full-grid
  search, 98.24% on the full 2018/2022 QPSK-like corpus, corrected by the `FFT{|z|^2}`
  cross-check arbitration added in `boi_p96_fft3` -- 0 regressions on any group-split
  partition, 936/167,816 samples recovered, raising accuracy 97.175%->97.965% (2018),
  99.309%->99.429% (2022), 97.355%->98.120% (2018R2); see `rs_prominence_fftz2_correction`'s
  docstring in `pre_boi_p96_fft2_generator.py`). The UNCORRECTED `rs_estimate_prominence` is
  still available in the upstream file for ablation via `CFG["RS_FIELD"]`.
- **CFO**: `cfo_estimate_band_interp` (band-limited + 3-point parabolic interpolation; verified
  this session to close the downstream classification gap to ground-truth CFO to within
  +/-0.4pp, vs. an 8-9pp gap with the old unrestricted-argmax CFO).

Either field can be swapped via `CFG["RS_FIELD"]` / `CFG["CFO_FIELD"]` (e.g. back to the
UNCORRECTED `rs_estimate_prominence`, or to `rs_estimate_abs_x2_rule` / `cfo_estimate_argmax`)
for ablation, without regenerating anything upstream.

The upstream files contain ALL signal types; the QPSK-like filter is applied HERE. Excluded
rows are counted separately in the output metadata, not silently dropped:
`num_upstream_not_qpsklike` (bpsk/8psk/dqpsk/msk rows), `num_upstream_negative_snr_skipped`
(2018 negative-SNR rows), and `num_upstream_error` (upstream BOI/clip/FFT failures).

`power.p_boi_inband_signal` (unclipped in-band power) is saved alongside the raw CC matrix for
the consumer to apply abs/warp/scale normalization later -- this notebook does not warp or scale.

**Prerequisite**: `generated_boi_p96_fft2_pt/2022_batch_NN_boi_p96_fft2_full.pt` must
already exist for `DATASET_SELECT="2022"` (run
`pre_BOI+FFT-Generate-2022-compact-vectors-final.ipynb` first). This notebook does
not check for that at import time, only when a batch is actually generated.

Output: `generated_qpsklike_cc_11x15_boi_p96_fft2_pt/2022_batch_NN_qpsklike_cc_1x11x15_boi_p96_fft2.pt`.


In [ ]:
from pathlib import Path
import sys


# Run with the Jupyter working directory set to this notebook's folder.
PROJECT_DIR = Path(".")
if not (PROJECT_DIR / "helper_function" / "__init__.py").is_file():
    raise FileNotFoundError("Run this notebook from the project directory containing helper_function.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from helper_function.pre_qpsklike_cc_11x15_boi_p96_fft2_generator import (
    generate_qpsklike_cc_pt_files,
    inspect_qpsklike_cc_pt,
    DEFAULT_RS_FIELD,
    DEFAULT_CFO_FIELD,
)

BASE = PROJECT_DIR
INPUT_ROOT = BASE / "generated_boi_p96_fft2_pt"
OUTPUT_ROOT = BASE / "generated_qpsklike_cc_11x15_boi_p96_fft2_pt"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("INPUT_ROOT:", INPUT_ROOT, "exists:", INPUT_ROOT.exists())
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("default RS_FIELD:", DEFAULT_RS_FIELD, "| default CFO_FIELD:", DEFAULT_CFO_FIELD)


## Generation Config


In [ ]:
CFG = {
    "INPUT_ROOT": str(INPUT_ROOT),
    "OUT_ROOT": str(OUTPUT_ROOT),
    "DATASET_SELECT": "2022",
    "N_BATCHES": 28,
    "START_BATCH": 1,
    "END_BATCH": 28,
    "SKIP_EXISTING": False,
    "DEVICE": "auto",
    "COMPUTE_BATCH_SIZE": 16,
    "RS_FIELD": DEFAULT_RS_FIELD,
    "CFO_FIELD": DEFAULT_CFO_FIELD,
}

print("Will generate to:", CFG["OUT_ROOT"])
print("Dataset select:", CFG["DATASET_SELECT"], "batches:", CFG["START_BATCH"], "to", CFG["END_BATCH"])
print("RS_FIELD:", CFG["RS_FIELD"], "| CFO_FIELD:", CFG["CFO_FIELD"])


## Run Generation

Performs the full `2022` CC generation from `generated_boi_p96_fft2_pt/`. Requires
that notebook's prerequisite generation to have already completed for this dataset.


In [ ]:
paths = generate_qpsklike_cc_pt_files(CFG)


## Inspect One Output File

Run after generation.


In [ ]:
inspect_qpsklike_cc_pt(paths[0])
